# PS4 v4: 3D hand tracking (Colab)
**Before you start**
1. Runtime → Change runtime type → **A100** (T4 also works, ~2–3× slower).
2. Upload **`ps4v4.zip`** to **`MyDrive/mytronD/`** (same folder as the clips).
3. The 40 clips (`MTL-....mp4`, `mytronlabs-....mp4`) are in `MyDrive/mytronD/` (change `VIDEO_DIR` in cell 3 if not).
4. In cell 3 set `PART = 1` on the first account and `PART = 2` on the second.

Run the cells top to bottom. Results appear in **`MyDrive/mytronD/ps4v4_results/<clip>/`** as each clip finishes.

*Data rule: the outputs contain footage. Keep them private and delete all copies after the event.*

In [ ]:
# CELL 1: install (~3 min)
!pip -q install ultralytics "smplx==0.1.28" timm einops roma scikit-image huggingface_hub pyyaml setuptools wheel
!pip -q install --no-deps git+https://github.com/warmshao/WiLoR-mini
!pip -q install --no-build-isolation --no-deps git+https://github.com/mattloper/chumpy
!pip -q install git+https://github.com/cvg/GeoCalib
import importlib.util, pathlib
_p = pathlib.Path(importlib.util.find_spec("chumpy").submodule_search_locations[0])
for _f in _p.glob("*.py"):                      # chumpy (MANO loading) breaks on new Python/NumPy
    _s = _f.read_text(); _o = _s
    _s = _s.replace("from numpy import bool, int, float, complex, object, unicode, str, nan, inf", "from numpy import nan, inf")
    _s = _s.replace("inspect.getargspec", "inspect.getfullargspec")
    if _s != _o: _f.write_text(_s)
import chumpy, torch
print("install OK | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU!")

In [ ]:
# CELL 2: mount Drive and unpack the code
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil, os
DRIVE_DIR = "/content/drive/MyDrive/mytronD"
ZIP = f"{DRIVE_DIR}/ps4v4.zip"
assert os.path.exists(ZIP), "upload ps4v4.zip to MyDrive/mytronD first"
shutil.rmtree("/content/ps4v4", ignore_errors=True); shutil.rmtree("/content/code", ignore_errors=True)
zipfile.ZipFile(ZIP).extractall("/content")
if os.path.exists("/content/code/run.py"):                       # new zips unpack to code/
    shutil.move("/content/code", "/content/ps4v4")
!cd /content/ps4v4 && python tests/test_synthetic.py 2>&1 | tail -3      # 1-min self-test, must end with ALL CHECKS PASSED
# download the model files ONCE here; parallel clips downloading at the same time crash each other
!cd /content/ps4v4 && python -c "from ps4v4.models.wilor_loader import load_wilor; load_wilor('cuda'); print('WiLoR weights OK')"
!python -c "from geocalib import GeoCalib; GeoCalib(); print('GeoCalib weights OK')"
!python -c "from transformers import CLIPModel, CLIPProcessor as P; CLIPModel.from_pretrained('openai/clip-vit-base-patch32'); P.from_pretrained('openai/clip-vit-base-patch32'); print('CLIP weights OK')"

In [ ]:
# CELL 3: run the clips of THIS account's half (results land in Drive as each clip finishes)
import glob, os, re, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed

PART      = 1                              # account 1: PART = 1 | account 2: PART = 2
N_PARTS   = 2
VIDEO_DIR = "/content/drive/MyDrive/mytronD"          # folder with the 40 MTL-... / mytronlabs-... clips
RESULTS   = f"{VIDEO_DIR}/ps4v4_results"
WORK      = "/content/ps4v4_work"          # fast local cache; finished files are copied to Drive
WORKERS   = 4                              # clips in parallel (A100: 4; drop to 3 if a clip fails out of memory)
MAX_SECONDS = None                         # e.g. 10 for a quick check
ARGS = ["camera.use_geocalib=true"]        # same settings as the runs you checked
KEEP = ["render.mp4", "hands.jsonl", "robot_wrist_gripper.csv", "stats.json", "camera.json", "tracklets.csv",
        "tracklets.pkl", "probe.pkl", "head_rot.pkl", "hands_mesh.npz", "quality.json"]
# Hand meshes for the SLAM world view: list clips here (e.g. ["MTL-SKL00089-006"]) to run ONLY those clips and
# add hands_mesh.npz to their existing result folders. Nothing else in those folders is changed.
MESH_FOR = []

def clip_name(p):                          # "Copy of Copy of MTL-SKL00089-011.mp4" -> "MTL-SKL00089-011"
    return os.path.splitext(os.path.basename(p))[0].replace("Copy of ", "").strip().replace(" ", "_")

all_clips = sorted((p for p in glob.glob(f"{VIDEO_DIR}/*.mp4")
                    if re.match(r"(MTL-|mytronlabs-)", clip_name(p))), key=clip_name)   # same order on any account
assert all_clips, "no MTL-... / mytronlabs-... .mp4 files in " + VIDEO_DIR
half = (len(all_clips) + N_PARTS - 1) // N_PARTS
clips = all_clips[(PART - 1) * half: PART * half]                 # PART 1 = first half, PART 2 = second half
if MESH_FOR:                                                      # mesh mode: just these clips, any account
    clips = [c for c in all_clips if clip_name(c) in MESH_FOR]
    missing = set(MESH_FOR) - {clip_name(c) for c in clips}
    assert not missing, f"no video for {missing} in {VIDEO_DIR}"
print(f"{len(all_clips)} clips found; this account (PART {PART}) runs {len(clips)}:")
print([clip_name(c) for c in clips])
os.makedirs(RESULTS, exist_ok=True); os.makedirs(WORK, exist_ok=True); os.makedirs("/content/videos", exist_ok=True)

def run_one(clip):
    name = clip_name(clip)
    out = f"{RESULTS}/{name}"
    mesh_mode = bool(MESH_FOR)
    if mesh_mode and os.path.exists(f"{out}/hands_mesh.npz"):
        return name, "already has hands_mesh.npz"
    if not mesh_mode and os.path.exists(f"{out}/hands.jsonl") and os.path.exists(f"{out}/stats.json"):
        return name, "already done (delete its results folder to re-run)"
    local = f"/content/videos/{name}.mp4"
    if not os.path.exists(local):
        shutil.copy(clip, local)                      # reading from local disk is much faster than Drive
    sets = [f"paths.cache_root={WORK}", f"max_seconds={'null' if MAX_SECONDS is None else MAX_SECONDS}", *ARGS]
    t0 = time.time()
    log = f"{WORK}/{name}.log"
    with open(log, "w") as fh:
        rc = subprocess.run([sys.executable, "run.py", "--clip", local, "--set", *sets],
                            cwd="/content/ps4v4", stdout=fh, stderr=subprocess.STDOUT).returncode
    os.makedirs(out, exist_ok=True)
    shutil.copy(log, f"{out}/run_mesh.log" if MESH_FOR else f"{out}/run.log")
    if rc != 0:
        return name, f"FAILED (see {out}/run.log)"
    keep = KEEP
    if mesh_mode and os.path.exists(f"{out}/hands.jsonl"):       # keep the checked results, add only the mesh
        keep = ["hands_mesh.npz"]
        if not os.path.exists(f"{WORK}/{name}/hands_mesh.npz"):
            return name, f"FAILED: no hands_mesh.npz written (see {out}/run.log)"
    for f in keep:
        src = f"{WORK}/{name}/{f}"
        if os.path.exists(src):
            shutil.copy(src, f"{out}/{f}")
    try:
        os.remove(local)                              # free local disk
    except OSError:
        pass
    return name, f"done in {(time.time() - t0) / 60:.1f} min"

t_all = time.time()
done = 0
with ThreadPoolExecutor(WORKERS) as ex:
    for fut in as_completed([ex.submit(run_one, c) for c in clips]):
        name, msg = fut.result()
        done += 1
        print(f"[{(time.time() - t_all) / 60:5.1f} min] {done}/{len(clips)} {name}: {msg}", flush=True)
print("all done. Re-run this cell to retry any FAILED clip (finished ones are skipped).")

In [ ]:
# CELL 4: summary table + one render
import json, glob, os
import pandas as pd
from IPython.display import Video, display
rows = []
for d in sorted(glob.glob(f"{RESULTS}/*")):
    p = f"{d}/stats.json"
    if not os.path.exists(p):
        continue
    s = json.load(open(p))
    rows.append({"clip": os.path.basename(d), "focal": round(s["camera"]["fx"]), "z_ref_m": s.get("z_ref_m"),
                 **{f"{k}_tracklets": v for k, v in s.get("tracklets", {}).items()},
                 **{f"{sd}_observed": s[sd]["share_observed"] for sd in ("left", "right")},
                 **{f"{sd}_estimated": s[sd]["share_estimated_out_of_view"] for sd in ("left", "right")},
                 "minutes": round(sum(s["seconds"].values()) / 60, 1)})
display(pd.DataFrame(rows))
SHOW = rows[0]["clip"] if rows else ""          # which render to play here (any clip name)
display(Video(f"{RESULTS}/{SHOW}/render.mp4", embed=True, width=960))

### Tuning without re-running everything
Every tracklet's evidence and decision is in `tracklets.csv` (columns `hand`, `side`, `label`, `reasons`).
Rejected tracklets show in the render as grey boxes with the reason (`static`, `far x2.4`, `arm away`, `kp/box`, `side taken`, `short`).
Changing only `assign.*` settings re-runs just the decision step and later steps (a minute or two per clip), as long as `/content/ps4v4_work` still exists:

In [ ]:
# CELL 5 (optional): re-decide one clip with different assign settings, e.g. stricter
CLIP = "MTL-SGF00106-021"
import glob, shutil, os
if not os.path.exists(f"/content/videos/{CLIP}.mp4"):
    shutil.copy([p for p in glob.glob(f"{VIDEO_DIR}/*.mp4") if CLIP in p][0], f"/content/videos/{CLIP}.mp4")
!cd /content/ps4v4 && python run.py --clip /content/videos/{CLIP}.mp4 --set paths.cache_root={WORK} camera.use_geocalib=true assign.accept_bias=0.3
!cp {WORK}/{CLIP}/render.mp4 {WORK}/{CLIP}/tracklets.csv {WORK}/{CLIP}/stats.json {RESULTS}/{CLIP}/